In [0]:
import dlt
from pyspark.sql.types import StructType,StructField,IntegerType,DoubleType,StringType,TimestampType
import pyspark.sql.functions as F

In [0]:
input_schema_raw=StructType([StructField('CRASH DATE', StringType(), True), StructField('CRASH TIME', StringType(), True), StructField('BOROUGH', StringType(), True), StructField('ZIP CODE', StringType(), True), StructField('LATITUDE', DoubleType(), True), StructField('LONGITUDE', DoubleType(), True), StructField('LOCATION', StringType(), True), StructField('ON STREET NAME', StringType(), True), StructField('CROSS STREET NAME', StringType(), True), StructField('OFF STREET NAME', StringType(), True), StructField('NUMBER OF PERSONS INJURED', StringType(), True), StructField('NUMBER OF PERSONS KILLED', IntegerType(), True), StructField('NUMBER OF PEDESTRIANS INJURED', IntegerType(), True), StructField('NUMBER OF PEDESTRIANS KILLED', IntegerType(), True), StructField('NUMBER OF CYCLIST INJURED', IntegerType(), True), StructField('NUMBER OF CYCLIST KILLED', StringType(), True), StructField('NUMBER OF MOTORIST INJURED', StringType(), True), StructField('NUMBER OF MOTORIST KILLED', IntegerType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 1', StringType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 2', StringType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 3', StringType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 4', StringType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 5', StringType(), True), StructField('COLLISION ID', IntegerType(), True), StructField('VEHICLE TYPE CODE 1', StringType(), True), StructField('VEHICLE TYPE CODE 2', StringType(), True), StructField('VEHICLE TYPE CODE 3', StringType(), True), StructField('VEHICLE TYPE CODE 4', StringType(), True), StructField('VEHICLE TYPE CODE 5', StringType(), True)])

In [0]:
file_path= spark.conf.get('source_path')
@dlt.table(name='vehicle_accidents_stream',
           table_properties={'quality': 'bronze', 'delta.columnMapping.mode': 'name',
   'delta.minReaderVersion' : '3',   'delta.minWriterVersion' : '7'})

def bronze_vehicle_crashes():
  return spark.readStream.format('cloudFiles')\
     .option("cloudFiles.format", "csv")\
     .option('header','true')\
     .schema(input_schema_raw)\
     .option("recursiveFileLookup", "true")\
     .load(file_path)\
     .withColumn('Bronze_Ingestion_Timestamp',F.current_timestamp())


In [0]:
input_schema_silver=StructType([StructField('CRASH_DATE', StringType(), True), StructField('CRASH_TIME', StringType(), True), StructField('BOROUGH', StringType(), True), StructField('ZIP_CODE', StringType(), True), StructField('LATITUDE', DoubleType(), True), StructField('LONGITUDE', DoubleType(), True), StructField('LOCATION', StringType(), True), StructField('ON_STREET_NAME', StringType(), True), StructField('CROSS_STREET_NAME', StringType(), True), StructField('OFF_STREET_NAME', StringType(), True), StructField('NUMBER_OF_PERSONS_INJURED', StringType(), True), StructField('NUMBER_OF_PERSONS_KILLED', IntegerType(), True), StructField('NUMBER_OF_PEDESTRIANS_INJURED', IntegerType(), True), StructField('NUMBER_OF_PEDESTRIANS_KILLED', IntegerType(), True), StructField('NUMBER_OF_CYCLIST_INJURED', IntegerType(), True), StructField('NUMBER_OF_CYCLIST_KILLED', StringType(), True), StructField('NUMBER_OF_MOTORIST_INJURED', StringType(), True), StructField('NUMBER_OF_MOTORIST_KILLED', IntegerType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_1', StringType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_2', StringType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_3', StringType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_4', StringType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_5', StringType(), True), StructField('COLLISION_ID', IntegerType(), True), StructField('VEHICLE_TYPE_CODE_1', StringType(), True), StructField('VEHICLE_TYPE_CODE_2', StringType(), True), StructField('VEHICLE_TYPE_CODE_3', StringType(), True), StructField('VEHICLE_TYPE_CODE_4', StringType(), True), StructField('VEHICLE_TYPE_CODE_5', StringType(), True),
 StructField('Bronze_Ingestion_Timestamp', TimestampType(), True)])

In [0]:
@dlt.table(name='silver.vehicle_accidents_cleansed_stream', table_properties={'schema': 'silver'})
def vehicle_accidents_cleansed_stream():
    df = dlt.read_stream('vehicle_accidents_stream')
    for clm in df.schema:
        col_name=clm.name
        col_new_name=col_name.replace(' ', '_')
        col_type=input_schema_silver[col_new_name].dataType
        df = df.withColumn(col_new_name,F.col(col_name).cast(col_type))
        if col_name!= col_new_name:
            df=df.drop(col_name)
    df=df.withColumn('ACCIDENT_DATE_TIME', F.to_timestamp(\
        F.concat(F.col('CRASH_DATE'), F.lit(' '), F.lpad(F.col('CRASH_TIME'), 5, '0')), 'MM/dd/yyyy HH:mm'))\
        .drop('LATITUDE','LONGITUDE','CRASH_DATE','CRASH_TIME')\
        .filter(F.col('ACCIDENT_DATE_TIME').isNotNull())    
    
    return df

1. Streaming from storage
2. Autoloader
3. Streaming from EH

## Reading from Event Hubs

In [0]:
connection_string_ehs = dbutils.secrets.get(scope = "fikrats_study_scope", key = "eh-dbr-source-connstr")
event_hub_namespace="eh-dbr"
source_event_hub_name="dbr-source"
sink_event_hub_name="dbr-target"

In [0]:
import dlt
import pyspark.sql.types as T
from pyspark.sql.functions import *

# Event Hubs configuration
EH_NAMESPACE                    = event_hub_namespace
EH_NAME                         = source_event_hub_name
EH_CONN_STR                     = connection_string_ehs
# Kafka Consumer configuration

KAFKA_OPTIONS = {
  "kafka.bootstrap.servers"  : f"{EH_NAMESPACE}.servicebus.windows.net:9093",
  "subscribe"                : EH_NAME,
  "kafka.sasl.mechanism"     : "PLAIN",
  "kafka.security.protocol"  : "SASL_SSL",
  "kafka.sasl.jaas.config"   : f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username=\"$ConnectionString\" password=\"{EH_CONN_STR}\";"
}

# PAYLOAD SCHEMA

@dlt.table()
def audit_raw():
  return (
   spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
  )

In [0]:
import pyspark.sql.functions as F 
@dlt.table(name='silver.audit_cleansed')
def audit_cleansed():
  df = dlt.read_stream('audit_raw')
  return (df.withColumnRenamed('enqueuedTime', 'timestamp')\
      .withColumn("records", col("value").cast("string")))\
      .drop("value")  
      